In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings("ignore")


def find_project_root(start=Path.cwd()):
    """Find the repository from either its root or this analysis folder."""
    for candidate in (start.resolve(), *start.resolve().parents):
        data_file = candidate / "1init_data" / "data" / "MutSpecVertebrates12.csv.gz"
        if data_file.is_file():
            return candidate
    raise FileNotFoundError("Could not locate the project root")


ROOT = find_project_root()
DATA_PATH = ROOT / "1init_data" / "data" / "MutSpecVertebrates12.csv.gz"

In [ ]:
mutspecs = pd.read_csv(DATA_PATH)
mutspecs = mutspecs.loc[mutspecs["Class"].eq("Mammalia")].copy()
mutspecs["Observed"] = mutspecs["Observed"].fillna(0)
mutspecs.head()

In [ ]:
mutspecs.Gene.value_counts()

In [ ]:
# Function to translate mutspec from light to heavy chain
def getrevers(mut):
    translator = str.maketrans("ACGT", "TGCA")
    #new_mut = mut[-1] +  mut[1:-1] + mut[0]
    new_mut = mut.translate(translator)
    return new_mut

In [ ]:
mutspecs['Mut'] = mutspecs.Mut.apply(getrevers)
#mutspecs["MutBase"] = mutspecs.Mut.str.slice(2, 5)
#mutspecs["Context"] = mutspecs.Mut.str.get(0) + mutspecs.Mut.str.get(2) + mutspecs.Mut.str.get(-1)
mutspecs.head()

### Make sum specific by each species and gene 

In [ ]:
mutspecs['MutSpec12'] = mutspecs.groupby(['Species', 'Gene', "Mut"])['MutSpec'].transform('sum')

In [ ]:
mutspecs.head()

### Check that Mutspec is on a heavy chain (for example only CytB gene)
Most common 12 comp Mutation are C>T and A>G, so it's heavy chain 

In [ ]:
mutspecs.loc[mutspecs.Gene.eq('Cytb')].groupby(['Mut']).MutSpec12.mean().reset_index().sort_values(by='MutSpec12', ascending=False).head(5)

In [ ]:
_ct_mt = mutspecs.loc[mutspecs['Mut'].eq('C>T')]
gene_order = ["ND2", "CO1", "CO3", "Cytb"]

In [ ]:
plt.figure(figsize=(13, 6))

ax = sns.boxplot(
    data=_ct_mt,
    x="Gene", y="MutSpec",
    order=gene_order,
    notch=True,
    width=0.55,
    linewidth=1.1,
    showfliers=False,
    palette="Greens"
)

ax.set_xlabel("Genes", fontsize=11)
ax.set_ylabel("Mutational Spectra", fontsize=11)
ax.set_title("C>T mutational spectrum profile", fontsize=12)

sns.despine(ax=ax)

# Gene identity is shown directly on the x-axis.

plt.tight_layout()
plt.show()

### Filter dataset to make sure that each gene is represented by common species

In [ ]:
species_gene_counts = _ct_mt.groupby("Species")["Gene"].nunique()

In [ ]:
common_species = species_gene_counts[species_gene_counts == len(gene_order)].index
df_common_CT = _ct_mt[_ct_mt["Species"].isin(common_species)].copy()

In [ ]:
plt.figure(figsize=(13, 6))

ax = sns.boxplot(
    data=df_common_CT,
    x="Gene", y="MutSpec",
    order=gene_order,
    notch=True,
    width=0.55,
    linewidth=1.1,
    showfliers=False,
    palette="Greens"
)

ax.set_xlabel("Genes", fontsize=11)
ax.set_ylabel("Mutational Spectra", fontsize=11)
ax.set_title(
    f"C>T mutational spectrum profile for species shared by all genes\n(N={len(common_species)})",
    fontsize=12,
)

sns.despine(ax=ax)

ax.legend(title="", frameon=False, loc="upper right")

plt.tight_layout()
plt.show()